In [ ]:
from netgen.geom2d import unit_square
from ngsolve import *
import numpy as np

# Problem setup (edit as needed)
Gamma_w = "right|top|bottom"
Sstor   = 1.0
p_init  = 0.0

# Permeability bump
k0    = 1.0
delta = 10.0
w     = 0.06

# Forcing Gaussian
f_amp   = 1.0
f_mu_x  = 0.3
f_mu_y  = 0.5
f_sigma = 0.2

# Exponential ramp settings
t_ramp_end  = 0.25
ramp_target = 0.6321205588
tau_ramp    = -t_ramp_end / np.log(1.0 - ramp_target)


def ramp(t: float) -> float:
    t = float(t)
    if t <= 0.0:
        return 0.0
    return 1.0 - np.exp(-t / tau_ramp)


def forcing_cf_time(t: float):
    base = f_amp * exp(-((x-f_mu_x)**2 + (y-f_mu_y)**2) / (2*f_sigma**2))
    return ramp(t) * base


def build_solver(mesh, order, dt, mu, inverse="pardiso"):
    mu_x, mu_y = float(mu[0]), float(mu[1])
    V = HDiv(mesh, order=int(order), dirichlet=Gamma_w)
    Q = L2(mesh, order=int(order)-1)
    Y = FESpace([V, Q])

    (uY, pY) = Y.TrialFunction()
    (vY, qY) = Y.TestFunction()

    K = k0 + delta * exp(-((x-mu_x)**2 + (y-mu_y)**2) / (2*w**2))
    Kinv = 1.0 / K
    S_cf = CoefficientFunction(float(Sstor))

    gfu = GridFunction(Y)
    uh, ph = gfu.components

    p_old = GridFunction(Q)
    p_old.Set(CoefficientFunction(float(p_init)))

    a = BilinearForm(Y, symmetric=True)
    a += (
        Kinv * InnerProduct(uY, vY)
        - pY * div(vY)
        + qY * div(uY)
        + (S_cf/dt) * pY * qY
    ) * dx
    a.Assemble()

    invA = a.mat.Inverse(Y.FreeDofs(), inverse=inverse)
    return V, Q, Y, invA, S_cf, gfu, uh, ph, p_old


def solve_history(mesh, order, dt, T, mu, inverse="pardiso"):
    ratio = T / dt
    nsteps = int(np.rint(ratio))
    tol = 1e-10 * max(1.0, abs(ratio))
    if abs(ratio - nsteps) > tol:
        raise ValueError(f"T/dt must be integer (within floating tolerance); got T={T}, dt={dt}, ratio={ratio}")

    V, Q, Y, invA, S_cf, gfu, uh, ph, p_old = build_solver(mesh, order, dt, mu, inverse=inverse)
    (_, _), (_, qY) = Y.TrialFunction(), Y.TestFunction()

    Nu, Np = V.ndof, Q.ndof
    Uhist = np.empty((nsteps+1, Nu), dtype=np.float64)
    Phist = np.empty((nsteps+1, Np), dtype=np.float64)

    Uhist[0, :] = 0.0
    Phist[0, :] = p_old.vec.FV().NumPy()

    for n in range(1, nsteps+1):
        t = n * dt
        f_cf = forcing_cf_time(t)

        L = LinearForm(Y)
        L += (f_cf * qY + (S_cf/dt) * p_old * qY) * dx
        L.Assemble()

        gfu.vec.data = invA * L.vec
        p_old.vec.data = ph.vec

        Uhist[n, :] = uh.vec.FV().NumPy()
        Phist[n, :] = ph.vec.FV().NumPy()

    return V, Q, Uhist, Phist


def gf_from_vec(space, vec):
    g = GridFunction(space)
    g.vec.FV().NumPy()[:] = vec
    return g


def sampled_l2_pressure_error_and_ref_norm(mesh_num, p_num, mesh_ref, p_ref, npts=121, eps=1e-10):
    xs = np.linspace(eps, 1.0-eps, int(npts))
    ys = np.linspace(eps, 1.0-eps, int(npts))

    e2_p = 0.0
    r2_p = 0.0
    cnt = 0

    for xv in xs:
        for yv in ys:
            mip_num = mesh_num(xv, yv)
            mip_ref = mesh_ref(xv, yv)
            if mip_num is None or mip_ref is None:
                continue

            pv_num = float(p_num(mip_num))
            pv_ref = float(p_ref(mip_ref))
            dp = pv_num - pv_ref

            e2_p += dp*dp
            r2_p += pv_ref*pv_ref
            cnt += 1

    if cnt == 0:
        raise RuntimeError("No sampling points inside both meshes")

    abs_p_err = np.sqrt(e2_p / cnt)
    ref_p_norm = np.sqrt(r2_p / cnt)
    return abs_p_err, ref_p_norm


def pressure_l2_in_time_relative_error(
    mesh_num, Q_num, Phist_num,
    mesh_ref, Q_ref, Phist_ref,
    stride_ref=1,
    npts=121,
):
    n_num = int(Phist_num.shape[0])
    n_ref = int(Phist_ref.shape[0])
    expected_ref = (n_num - 1) * int(stride_ref) + 1
    if expected_ref != n_ref:
        raise ValueError(
            f"Incompatible history sizes for L2-in-time error: "
            f"n_num={n_num}, n_ref={n_ref}, stride_ref={stride_ref}"
        )

    e2_t = 0.0
    r2_t = 0.0

    for n in range(n_num):
        p_num = gf_from_vec(Q_num, Phist_num[n, :])
        p_ref = gf_from_vec(Q_ref, Phist_ref[n*stride_ref, :])
        abs_err_n, ref_norm_n = sampled_l2_pressure_error_and_ref_norm(
            mesh_num, p_num, mesh_ref, p_ref, npts=npts
        )
        e2_t += abs_err_n * abs_err_n
        r2_t += ref_norm_n * ref_norm_n

    abs_p_l2_t = np.sqrt(e2_t / n_num)
    ref_p_l2_t = np.sqrt(r2_t / n_num)
    rel_p_l2_t = abs_p_l2_t / ref_p_l2_t if ref_p_l2_t > 1e-30 else abs_p_l2_t
    return abs_p_l2_t, rel_p_l2_t


def sampled_l2_errors(mesh_num, p_num, u_num, mesh_ref, p_ref, u_ref, npts=121, eps=1e-10):
    xs = np.linspace(eps, 1.0-eps, int(npts))
    ys = np.linspace(eps, 1.0-eps, int(npts))

    e2_p = 0.0
    r2_p = 0.0
    e2_u = 0.0
    r2_u = 0.0
    cnt = 0

    for xv in xs:
        for yv in ys:
            mip_num = mesh_num(xv, yv)
            mip_ref = mesh_ref(xv, yv)
            if mip_num is None or mip_ref is None:
                continue

            pv_num = float(p_num(mip_num))
            pv_ref = float(p_ref(mip_ref))

            uv_num = np.array(u_num(mip_num), dtype=np.float64)
            uv_ref = np.array(u_ref(mip_ref), dtype=np.float64)

            dp = pv_num - pv_ref
            du = uv_num - uv_ref

            e2_p += dp*dp
            r2_p += pv_ref*pv_ref
            e2_u += float(np.dot(du, du))
            r2_u += float(np.dot(uv_ref, uv_ref))
            cnt += 1

    if cnt == 0:
        raise RuntimeError("No sampling points inside both meshes")

    abs_p = np.sqrt(e2_p / cnt)
    abs_u = np.sqrt(e2_u / cnt)

    rel_p = abs_p / np.sqrt(r2_p / cnt) if r2_p > 1e-30 else abs_p
    rel_u = abs_u / np.sqrt(r2_u / cnt) if r2_u > 1e-30 else abs_u

    return abs_p, rel_p, abs_u, rel_u


def sampled_velocity_l2_hdiv_error_and_ref_norm(mesh_num, u_num, mesh_ref, u_ref, npts=121, eps=1e-10):
    xs = np.linspace(eps, 1.0-eps, int(npts))
    ys = np.linspace(eps, 1.0-eps, int(npts))

    div_u_num = div(u_num)
    div_u_ref = div(u_ref)

    e2_l2 = 0.0
    r2_l2 = 0.0
    e2_hdiv = 0.0
    r2_hdiv = 0.0
    cnt = 0

    for xv in xs:
        for yv in ys:
            mip_num = mesh_num(xv, yv)
            mip_ref = mesh_ref(xv, yv)
            if mip_num is None or mip_ref is None:
                continue

            uv_num = np.array(u_num(mip_num), dtype=np.float64)
            uv_ref = np.array(u_ref(mip_ref), dtype=np.float64)
            du = uv_num - uv_ref

            dnum = float(np.asarray(div_u_num(mip_num), dtype=np.float64).reshape(-1)[0])
            dref = float(np.asarray(div_u_ref(mip_ref), dtype=np.float64).reshape(-1)[0])
            dd = dnum - dref

            l2_err_pt = float(np.dot(du, du))
            l2_ref_pt = float(np.dot(uv_ref, uv_ref))

            e2_l2 += l2_err_pt
            r2_l2 += l2_ref_pt
            e2_hdiv += l2_err_pt + dd*dd
            r2_hdiv += l2_ref_pt + dref*dref
            cnt += 1

    if cnt == 0:
        raise RuntimeError("No sampling points inside both meshes")

    abs_u_l2_err = np.sqrt(e2_l2 / cnt)
    ref_u_l2_norm = np.sqrt(r2_l2 / cnt)
    abs_u_hdiv_err = np.sqrt(e2_hdiv / cnt)
    ref_u_hdiv_norm = np.sqrt(r2_hdiv / cnt)

    return abs_u_l2_err, ref_u_l2_norm, abs_u_hdiv_err, ref_u_hdiv_norm


def velocity_l2_in_time_relative_errors(
    mesh_num, V_num, Uhist_num,
    mesh_ref, V_ref, Uhist_ref,
    stride_ref=1,
    npts=121,
):
    n_num = int(Uhist_num.shape[0])
    n_ref = int(Uhist_ref.shape[0])
    expected_ref = (n_num - 1) * int(stride_ref) + 1
    if expected_ref != n_ref:
        raise ValueError(
            f"Incompatible velocity history sizes for L2-in-time error: "
            f"n_num={n_num}, n_ref={n_ref}, stride_ref={stride_ref}"
        )

    e2_l2_t = 0.0
    r2_l2_t = 0.0
    e2_hdiv_t = 0.0
    r2_hdiv_t = 0.0

    for n in range(n_num):
        u_num = gf_from_vec(V_num, Uhist_num[n, :])
        u_ref = gf_from_vec(V_ref, Uhist_ref[n*stride_ref, :])
        abs_l2_n, ref_l2_n, abs_hdiv_n, ref_hdiv_n = sampled_velocity_l2_hdiv_error_and_ref_norm(
            mesh_num, u_num, mesh_ref, u_ref, npts=npts
        )

        e2_l2_t += abs_l2_n * abs_l2_n
        r2_l2_t += ref_l2_n * ref_l2_n
        e2_hdiv_t += abs_hdiv_n * abs_hdiv_n
        r2_hdiv_t += ref_hdiv_n * ref_hdiv_n

    abs_u_l2_t = np.sqrt(e2_l2_t / n_num)
    ref_u_l2_t = np.sqrt(r2_l2_t / n_num)
    abs_u_hdiv_t = np.sqrt(e2_hdiv_t / n_num)
    ref_u_hdiv_t = np.sqrt(r2_hdiv_t / n_num)

    rel_u_l2_t = abs_u_l2_t / ref_u_l2_t if ref_u_l2_t > 1e-30 else abs_u_l2_t
    rel_u_hdiv_t = abs_u_hdiv_t / ref_u_hdiv_t if ref_u_hdiv_t > 1e-30 else abs_u_hdiv_t

    return abs_u_l2_t, rel_u_l2_t, abs_u_hdiv_t, rel_u_hdiv_t


def to_latex_sci(x, nd=1):
    if x == 0.0:
        return "$0$"
    s = f"{x:.{nd}e}"
    m, e = s.split("e")
    return f"${m}\\times 10^{{{int(e)}}}$"


def latex_table_from_grid(grid, maxh_list, dt_list, key, caption, label, nd=1):
    cols = "c" * len(dt_list)
    lines = []
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{{label}}}")
    lines.append(f"\\begin{{tabular}}{{l{cols}}}")
    lines.append("\\hline")
    header = "$h_{\\text{max}}$ / $\\Delta t$"
    for dt in dt_list:
        header += f" & {dt:g}"
    header += " \\\\" 
    lines.append(header)
    lines.append("\\hline")

    for h in maxh_list:
        row = f"{h:g}"
        for dt in dt_list:
            row += " & " + to_latex_sci(grid[(float(h), float(dt))][key], nd=nd)
        row += " \\\\" 
        lines.append(row)

    lines.append("\\hline")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")
    return "\n".join(lines)


def run_space_time_study(
    mu=(0.5, 0.5),
    T=1.0,
    order=2,
    maxh_list=(0.10, 0.05),
    dt_list=(0.05, 0.01),
    maxh_ref=0.025,
    dt_ref=0.0025,
    sample_npts=121,
    inverse="pardiso",
    latex_only=True,
    include_individual=False,
):
    maxh_list = [float(h) for h in maxh_list]
    dt_list = [float(dt) for dt in dt_list]

    mesh_ref = Mesh(unit_square.GenerateMesh(maxh=float(maxh_ref)))
    Vref, Qref, Uref_hist, Pref_hist = solve_history(mesh_ref, order, dt_ref, T, mu, inverse=inverse)

    if not latex_only:
        print("=== High-fidelity reference ===")
        print(f"mu={mu}, T={T}, order={order}, maxh_ref={maxh_ref}, dt_ref={dt_ref}")
        print(f"sampling grid: {sample_npts} x {sample_npts}\n")

    time_rows = []
    space_rows = []

    if include_individual:
        if not latex_only:
            print("=== Time study (fixed space) ===")

        mesh_time = Mesh(unit_square.GenerateMesh(maxh=float(maxh_ref)))
        _, _, Utime_ref, Ptime_ref = solve_history(mesh_time, order, dt_ref, T, mu, inverse=inverse)

        for dt in dt_list:
            ratio = dt / dt_ref
            rint = int(np.rint(ratio))
            tol = 1e-10 * max(1.0, abs(ratio))
            if abs(ratio - rint) > tol:
                raise ValueError(f"Each dt must be integer multiple of dt_ref (within floating tolerance); got dt={dt}, dt_ref={dt_ref}, ratio={ratio}")

            Vt, Qt, Ut_hist, Pt_hist = solve_history(mesh_time, order, dt, T, mu, inverse=inverse)

            abs_p, rel_p = pressure_l2_in_time_relative_error(
                mesh_time, Qt, Pt_hist,
                mesh_time, Qt, Ptime_ref,
                stride_ref=rint,
                npts=sample_npts,
            )

            abs_u_l2, rel_u_l2, abs_u_hdiv, rel_u_hdiv = velocity_l2_in_time_relative_errors(
                mesh_time, Vt, Ut_hist,
                mesh_time, Vt, Utime_ref,
                stride_ref=rint,
                npts=sample_npts,
            )

            time_rows.append((dt, abs_p, rel_p, abs_u_l2, rel_u_l2, abs_u_hdiv, rel_u_hdiv))
            if not latex_only:
                print(
                    f"dt={dt:g}: rel_p_l2_t={rel_p:.3e}, "
                    f"rel_u_l2_t={rel_u_l2:.3e}, rel_u_hdiv_l2_t={rel_u_hdiv:.3e}"
                )

        if not latex_only:
            print("\n=== Space study (fixed time step) ===")

        for h in maxh_list:
            mesh_h = Mesh(unit_square.GenerateMesh(maxh=float(h)))
            Vh, Qh, Uh_hist, Ph_hist = solve_history(mesh_h, order, dt_ref, T, mu, inverse=inverse)
            abs_p, rel_p = pressure_l2_in_time_relative_error(
                mesh_h, Qh, Ph_hist,
                mesh_ref, Qref, Pref_hist,
                stride_ref=1,
                npts=sample_npts,
            )

            abs_u_l2, rel_u_l2, abs_u_hdiv, rel_u_hdiv = velocity_l2_in_time_relative_errors(
                mesh_h, Vh, Uh_hist,
                mesh_ref, Vref, Uref_hist,
                stride_ref=1,
                npts=sample_npts,
            )

            space_rows.append((h, abs_p, rel_p, abs_u_l2, rel_u_l2, abs_u_hdiv, rel_u_hdiv))
            if not latex_only:
                print(
                    f"maxh={h:g}: rel_p_l2_t={rel_p:.3e}, "
                    f"rel_u_l2_t={rel_u_l2:.3e}, rel_u_hdiv_l2_t={rel_u_hdiv:.3e}"
                )

    if not latex_only:
        print("=== Joint (space, time) study ===")

    grid = {}
    for h in maxh_list:
        mesh_h = Mesh(unit_square.GenerateMesh(maxh=float(h)))
        for dt in dt_list:
            Vh, Qh, Uh_hist, Ph_hist = solve_history(mesh_h, order, dt, T, mu, inverse=inverse)

            ratio = dt / dt_ref
            rint = int(np.rint(ratio))
            tol = 1e-10 * max(1.0, abs(ratio))
            if abs(ratio - rint) > tol:
                raise ValueError(
                    f"Each dt must be integer multiple of dt_ref (within floating tolerance); "
                    f"got dt={dt}, dt_ref={dt_ref}, ratio={ratio}"
                )

            abs_p, rel_p = pressure_l2_in_time_relative_error(
                mesh_h, Qh, Ph_hist,
                mesh_ref, Qref, Pref_hist,
                stride_ref=rint,
                npts=sample_npts,
            )

            abs_u_l2, rel_u_l2, abs_u_hdiv, rel_u_hdiv = velocity_l2_in_time_relative_errors(
                mesh_h, Vh, Uh_hist,
                mesh_ref, Vref, Uref_hist,
                stride_ref=rint,
                npts=sample_npts,
            )

            grid[(float(h), float(dt))] = {
                "abs_p": abs_p,
                "rel_p": rel_p,
                "abs_u_l2": abs_u_l2,
                "rel_u_l2": rel_u_l2,
                "abs_u_hdiv": abs_u_hdiv,
                "rel_u_hdiv": rel_u_hdiv,
                # Backward-compatibility aliases:
                "abs_u": abs_u_l2,
                "rel_u": rel_u_l2,
            }
            if not latex_only:
                print(
                    f"(maxh={h:g}, dt={dt:g}): rel_p_l2_t={rel_p:.3e}, "
                    f"rel_u_l2_t={rel_u_l2:.3e}, rel_u_hdiv_l2_t={rel_u_hdiv:.3e}"
                )

    def sci_tex(x, decimals=1):
        if abs(x) < 1e-300:
            return r'$0.0\times 10^{0}$'
        s = f"{x:.{decimals}e}"
        mant, exp = s.split('e')
        exp_i = int(exp)
        return rf'${mant}\times 10^{{{exp_i}}}$'

    rows = []
    for (h, dt), vals in sorted(grid.items(), key=lambda kv: (-kv[0][0], -kv[0][1])):
        rows.append((h, dt, vals['rel_p'], vals['rel_u_l2'], vals['rel_u_hdiv']))

    lines = []
    lines.append(r'\begin{tabular}{ccccc}')
    lines.append(r'\hline')
    lines.append(r'$h_{\max}$ & $\Delta t$ & $\|e_p\|_{L^2}$ & $\|e_{\mathbf{u}}\|_{l^2}$ & $\|e_{\mathbf{u}}\|_{hdiv}$ \\')
    lines.append(r'\hline')

    for h, dt, ep, eu_l2, eu_hdiv in rows:
        lines.append(
            f"{h:.3g} & {dt:.3g} & {sci_tex(ep, 1)} & {sci_tex(eu_l2, 1)} & {sci_tex(eu_hdiv, 1)} \\\\"
        )

    lines.append(r'\hline')
    lines.append(r'\end{tabular}')

    latex_joint = '\n'.join(lines)
    print(latex_joint)

    return {
        "time_rows": time_rows,
        "space_rows": space_rows,
        "grid": grid,
        "latex_joint": latex_joint,
    }


if __name__ == "__main__":
    results = run_space_time_study(
        mu=(0.5, 0.5),
        T=1.0,
        order=2,
        maxh_list=(0.32,0.16,0.08,),
        dt_list=(0.04,0.02,0.01,0.005,),
        maxh_ref=0.04,
        dt_ref=0.00125,
        sample_npts=81,
        inverse="pardiso",
        latex_only=False,
        include_individual=False,
    )

=== High-fidelity reference ===
mu=(0.5, 0.5), T=1.0, order=2, maxh_ref=0.04, dt_ref=0.00125
sampling grid: 81 x 81

=== Joint (space, time) study ===
(maxh=0.32, dt=0.04): rel_p_l2_t=2.760e-02, rel_u_l2_t=3.102e-02, rel_u_hdiv_l2_t=1.107e-01
(maxh=0.32, dt=0.02): rel_p_l2_t=2.485e-02, rel_u_l2_t=2.880e-02, rel_u_hdiv_l2_t=1.106e-01
(maxh=0.32, dt=0.01): rel_p_l2_t=2.412e-02, rel_u_l2_t=2.810e-02, rel_u_hdiv_l2_t=1.106e-01
(maxh=0.32, dt=0.005): rel_p_l2_t=2.398e-02, rel_u_l2_t=2.789e-02, rel_u_hdiv_l2_t=1.106e-01
(maxh=0.16, dt=0.04): rel_p_l2_t=1.537e-02, rel_u_l2_t=1.358e-02, rel_u_hdiv_l2_t=2.259e-02
(maxh=0.16, dt=0.02): rel_p_l2_t=9.027e-03, rel_u_l2_t=7.767e-03, rel_u_hdiv_l2_t=2.172e-02
(maxh=0.16, dt=0.01): rel_p_l2_t=6.305e-03, rel_u_l2_t=5.101e-03, rel_u_hdiv_l2_t=2.146e-02
(maxh=0.16, dt=0.005): rel_p_l2_t=5.455e-03, rel_u_l2_t=4.191e-03, rel_u_hdiv_l2_t=2.139e-02
(maxh=0.08, dt=0.04): rel_p_l2_t=1.454e-02, rel_u_l2_t=1.298e-02, rel_u_hdiv_l2_t=9.152e-03
(maxh=0.08, dt=0.02

In [6]:
if __name__ == "__main__":
    results = run_space_time_study(
        mu=(0.3, 0.7),
        T=1.0,
        order=2,
        maxh_list=(0.32,0.16,0.08,),
        dt_list=(0.04,0.02,0.01,0.005,),
        maxh_ref=0.04,
        dt_ref=0.00125,
        sample_npts=81,
        inverse="pardiso",
        latex_only=False,
        include_individual=False,
    )

=== High-fidelity reference ===
mu=(0.3, 0.7), T=1.0, order=2, maxh_ref=0.04, dt_ref=0.00125
sampling grid: 81 x 81

=== Joint (space, time) study ===
(maxh=0.32, dt=0.04): rel_p_l2_t=3.061e-02, rel_u_l2_t=8.375e-02, rel_u_hdiv_l2_t=1.122e-01
(maxh=0.32, dt=0.02): rel_p_l2_t=2.820e-02, rel_u_l2_t=8.316e-02, rel_u_hdiv_l2_t=1.121e-01
(maxh=0.32, dt=0.01): rel_p_l2_t=2.753e-02, rel_u_l2_t=8.301e-02, rel_u_hdiv_l2_t=1.121e-01
(maxh=0.32, dt=0.005): rel_p_l2_t=2.738e-02, rel_u_l2_t=8.297e-02, rel_u_hdiv_l2_t=1.121e-01
(maxh=0.16, dt=0.04): rel_p_l2_t=1.542e-02, rel_u_l2_t=1.516e-02, rel_u_hdiv_l2_t=2.260e-02
(maxh=0.16, dt=0.02): rel_p_l2_t=9.480e-03, rel_u_l2_t=1.135e-02, rel_u_hdiv_l2_t=2.176e-02
(maxh=0.16, dt=0.01): rel_p_l2_t=7.057e-03, rel_u_l2_t=1.004e-02, rel_u_hdiv_l2_t=2.150e-02
(maxh=0.16, dt=0.005): rel_p_l2_t=6.338e-03, rel_u_l2_t=9.687e-03, rel_u_hdiv_l2_t=2.143e-02
(maxh=0.08, dt=0.04): rel_p_l2_t=1.426e-02, rel_u_l2_t=1.187e-02, rel_u_hdiv_l2_t=9.073e-03
(maxh=0.08, dt=0.02